# Day 14

## 3. Move paths, thresholds and mappings into config.yaml and read it in the notebook

In [2]:
import yaml
from pathlib import Path

with open("config.yaml", encoding="utf-8") as file:
    config = yaml.safe_load(file)

data_folder = Path(config["paths"]["data_folder"])
log_folder = Path(config["paths"]["log_folder"])
bad_records_file = Path(config["paths"]["bad_records_file"])
clean_csv_file = Path(config["paths"]["clean_csv_file"])
clean_json_file = Path(config["paths"]["clean_json_file"])
run_summary_file = Path(config["paths"]["run_summary_file"])
manifest_file = Path(config["paths"]["manifest_file"])
rejection_limit = config["rejection_limit"]
allowed_status = config["allowed_status"]
required_fields = config["required_fields"]

## 1. Replace print statements with logging (INFO, WARNING, ERROR)

In [3]:
import logging
import time
from datetime import datetime
from pathlib import Path

log_folder.mkdir(exist_ok=True)
log_file = log_folder / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s: %(message)s",
    handlers=[
        logging.FileHandler(log_file, encoding="utf-8"),
        logging.StreamHandler(),
    ],
    force=True,
)

start_time = time.perf_counter()
logging.info("Starting validation run")

2026-11-25 14:27:03,378 INFO: Starting validation run


## 2. Write a log file per run with a timestamp

In [4]:
import csv
import json
import re
from datetime import datetime
from pathlib import Path

bad_rows = []
clean_rows = []
file_summaries = []
processed_this_run = []

if manifest_file.is_file():
    with manifest_file.open(encoding="utf-8") as file:
        processed_files = json.load(file)
else:
    processed_files = []

files_to_process = []
for file_name in required_fields:
    if file_name in processed_files:
        logging.info("Skipping already processed file: %s", file_name)
    else:
        files_to_process.append(file_name)

for file_name in files_to_process:
    fields = required_fields[file_name]
    path = data_folder / file_name

    if not path.is_file():
        logging.warning("File is missing: %s", file_name)
        continue

    if path.stat().st_size == 0:
        logging.warning("File is empty: %s", file_name)
        continue

    with path.open("r", newline="", encoding="utf-8-sig") as file:
        first_line = file.readline()
        file.seek(0)
        delimiter = ";" if ";" in first_line else ","
        reader = csv.DictReader(file, delimiter=delimiter)

        if reader.fieldnames is None:
            logging.error("File has no header: %s", file_name)
            continue

        total = 0
        valid = 0
        rejected = 0

        for row_number, row in enumerate(reader, start=2):
            total += 1
            reasons = []

            for field in fields:
                if not row.get(field):
                    reasons.append(f"missing {field}")

            if "price" in row:
                try:
                    price = float(row["price"])
                    if price <= 0:
                        reasons.append("price must be greater than 0")
                except ValueError:
                    reasons.append("price is not numeric")

            if "order_status" in row and row["order_status"] not in allowed_status:
                reasons.append("invalid order_status")

            for field, value in row.items():
                if not value:
                    continue

                if "zip" in field or field.endswith("_prefix"):
                    if re.fullmatch(r"\d{5}", value) is None:
                        reasons.append(f"invalid zip prefix in {field}")

                if field.endswith("_id") and field != "order_item_id":
                    if re.fullmatch(r"[0-9a-fA-F]{32}", value) is None:
                        reasons.append(f"invalid ID in {field}")

                if "date" in field or "timestamp" in field or field.endswith("_at"):
                    try:
                        datetime.fromisoformat(value)
                    except ValueError:
                        reasons.append(f"invalid date in {field}")

            if reasons:
                rejected += 1
                bad_rows.append({
                    "file": file_name,
                    "row": row_number,
                    "reason": "; ".join(reasons),
                    "data": row,
                })
            else:
                valid += 1
                clean_row = {"file": file_name}
                clean_row.update(row)
                clean_rows.append(clean_row)

    if total == 0:
        logging.warning("File has no data rows: %s", file_name)
        continue

    logging.info("%s: total=%s, valid=%s, rejected=%s", file_name, total, valid, rejected)
    file_summaries.append({
        "file": file_name,
        "total": total,
        "valid": valid,
        "rejected": rejected,
    })
    processed_this_run.append(file_name)

    if rejected / total > rejection_limit:
        logging.error("More than %.0f%% of rows were rejected in %s", rejection_limit * 100, file_name)

if processed_this_run:
    with bad_records_file.open("w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["file", "row", "reason", "data"])

        for bad_row in bad_rows:
            writer.writerow([bad_row["file"], bad_row["row"], bad_row["reason"], bad_row["data"]])

    logging.info("Wrote %s rejected rows to bad_records.csv", len(bad_rows))
else:
    logging.info("No new files to process")

2026-11-25 14:27:03,406 INFO: olist_customers_dataset.csv: total=20, valid=13, rejected=7
2026-11-25 14:27:03,407 ERROR: More than 20% of rows were rejected in olist_customers_dataset.csv
2026-11-25 14:27:03,410 INFO: olist_geolocation_dataset.csv: total=20, valid=16, rejected=4
2026-11-25 14:27:03,412 INFO: olist_order_items_dataset.csv: total=20, valid=11, rejected=9
2026-11-25 14:27:03,412 ERROR: More than 20% of rows were rejected in olist_order_items_dataset.csv
2026-11-25 14:27:03,414 INFO: olist_orders_dataset.csv: total=20, valid=14, rejected=6
2026-11-25 14:27:03,416 ERROR: More than 20% of rows were rejected in olist_orders_dataset.csv
2026-11-25 14:27:03,416 INFO: olist_sellers_dataset.csv: total=20, valid=14, rejected=6
2026-11-25 14:27:03,416 ERROR: More than 20% of rows were rejected in olist_sellers_dataset.csv
2026-11-25 14:27:03,416 INFO: Wrote 32 rejected rows to bad_records.csv


## 4. Write the clean output as CSV and JSON

In [5]:
import json

if processed_this_run:
    output_folder = clean_csv_file.parent
    output_folder.mkdir(exist_ok=True)

    field_names = ["file"]
    for row in clean_rows:
        for field in row:
            if field not in field_names:
                field_names.append(field)

    with clean_csv_file.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=field_names)
        writer.writeheader()
        writer.writerows(clean_rows)

    with clean_json_file.open("w", encoding="utf-8") as file:
        json.dump(clean_rows, file, indent=2)

    logging.info("Wrote %s clean rows to CSV and JSON", len(clean_rows))
else:
    logging.info("No new clean files to write")

2026-11-25 14:27:03,432 INFO: Wrote 68 clean rows to CSV and JSON


## 5. Write a run summary JSON with counts and duration

In [6]:
total_rows = sum(summary["total"] for summary in file_summaries)
valid_rows = len(clean_rows)
rejected_rows = len(bad_rows)
duration = round(time.perf_counter() - start_time, 2)

run_summary = {
    "counts": {
        "total": total_rows,
        "valid": valid_rows,
        "rejected": rejected_rows,
    },
    "duration_seconds": duration,
    "files": file_summaries,
}

with run_summary_file.open("w", encoding="utf-8") as file:
    json.dump(run_summary, file, indent=2)

logging.info("Wrote run summary to %s", run_summary_file)

2026-11-25 14:27:03,446 INFO: Wrote run summary to output\run_summary.json


## 6. Keep a manifest of processed files and skip them next time

The validation cell skips filenames already in the manifest. This cell adds files processed during this run.

In [7]:
for file_name in processed_this_run:
    if file_name not in processed_files:
        processed_files.append(file_name)

manifest_file.parent.mkdir(exist_ok=True)
with manifest_file.open("w", encoding="utf-8") as file:
    json.dump(processed_files, file, indent=2)

logging.info("Saved %s processed filenames to the manifest", len(processed_files))

2026-11-25 14:27:03,457 INFO: Saved 5 processed filenames to the manifest


## 7. Run the whole pipeline from one notebook cell using config.yaml

In [8]:
import csv
import json
import logging
import re
import time
import yaml
from datetime import datetime
from pathlib import Path

start_time = time.perf_counter()

with open("config.yaml", encoding="utf-8") as file:
    config = yaml.safe_load(file)

data_folder = Path(config["paths"]["data_folder"])
log_folder = Path(config["paths"]["log_folder"])
bad_records_file = Path(config["paths"]["bad_records_file"])
clean_csv_file = Path(config["paths"]["clean_csv_file"])
clean_json_file = Path(config["paths"]["clean_json_file"])
run_summary_file = Path(config["paths"]["run_summary_file"])
manifest_file = Path(config["paths"]["manifest_file"])
rejection_limit = config["rejection_limit"]
allowed_status = set(config["allowed_status"])
required_fields = config["required_fields"]

log_folder.mkdir(exist_ok=True)
log_file = log_folder / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s: %(message)s",
    handlers=[
        logging.FileHandler(log_file, encoding="utf-8"),
        logging.StreamHandler(),
    ],
    force=True,
)

if manifest_file.is_file():
    with manifest_file.open(encoding="utf-8") as file:
        processed_files = json.load(file)
else:
    processed_files = []

bad_rows = []
clean_rows = []
file_summaries = []
processed_this_run = []

logging.info("Starting pipeline")

for file_name, fields in required_fields.items():
    if file_name in processed_files:
        logging.info("Skipping already processed file: %s", file_name)
        continue

    path = data_folder / file_name
    if not path.is_file():
        logging.warning("File is missing: %s", file_name)
        continue

    if path.stat().st_size == 0:
        logging.warning("File is empty: %s", file_name)
        continue

    with path.open("r", newline="", encoding="utf-8-sig") as file:
        first_line = file.readline()
        file.seek(0)
        delimiter = ";" if ";" in first_line else ","
        reader = csv.DictReader(file, delimiter=delimiter)

        if reader.fieldnames is None:
            logging.error("File has no header: %s", file_name)
            continue

        total = 0
        valid = 0
        rejected = 0

        for row_number, row in enumerate(reader, start=2):
            total += 1
            reasons = []

            for field in fields:
                if not row.get(field):
                    reasons.append(f"missing {field}")

            if "price" in row:
                try:
                    price = float(row["price"])
                    if price <= 0:
                        reasons.append("price must be greater than 0")
                except ValueError:
                    reasons.append("price is not numeric")

            if "order_status" in row and row["order_status"] not in allowed_status:
                reasons.append("invalid order_status")

            for field, value in row.items():
                if not value:
                    continue

                if "zip" in field or field.endswith("_prefix"):
                    if re.fullmatch(r"\d{5}", value) is None:
                        reasons.append(f"invalid zip prefix in {field}")

                if field.endswith("_id") and field != "order_item_id":
                    if re.fullmatch(r"[0-9a-fA-F]{32}", value) is None:
                        reasons.append(f"invalid ID in {field}")

                if "date" in field or "timestamp" in field or field.endswith("_at"):
                    try:
                        datetime.fromisoformat(value)
                    except ValueError:
                        reasons.append(f"invalid date in {field}")

            if reasons:
                rejected += 1
                bad_rows.append({
                    "file": file_name,
                    "row": row_number,
                    "reason": "; ".join(reasons),
                    "data": row,
                })
            else:
                valid += 1
                clean_row = {"file": file_name}
                clean_row.update(row)
                clean_rows.append(clean_row)

    if total == 0:
        logging.warning("File has no data rows: %s", file_name)
        continue

    file_summaries.append({
        "file": file_name,
        "total": total,
        "valid": valid,
        "rejected": rejected,
    })
    processed_this_run.append(file_name)

    logging.info("%s: total=%s, valid=%s, rejected=%s", file_name, total, valid, rejected)
    if rejected / total > rejection_limit:
        logging.error("More than %.0f%% of rows were rejected in %s", rejection_limit * 100, file_name)

if processed_this_run:
    bad_records_file.parent.mkdir(exist_ok=True)
    with bad_records_file.open("w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["file", "row", "reason", "data"])
        for bad_row in bad_rows:
            writer.writerow([bad_row["file"], bad_row["row"], bad_row["reason"], bad_row["data"]])

    field_names = ["file"]
    for row in clean_rows:
        for field in row:
            if field not in field_names:
                field_names.append(field)

    clean_csv_file.parent.mkdir(exist_ok=True)
    with clean_csv_file.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=field_names)
        writer.writeheader()
        writer.writerows(clean_rows)

    with clean_json_file.open("w", encoding="utf-8") as file:
        json.dump(clean_rows, file, indent=2)

    logging.info("Wrote %s clean and %s rejected rows", len(clean_rows), len(bad_rows))
else:
    logging.info("No new files to process")

run_summary = {
    "counts": {
        "total": sum(summary["total"] for summary in file_summaries),
        "valid": len(clean_rows),
        "rejected": len(bad_rows),
    },
    "duration_seconds": round(time.perf_counter() - start_time, 2),
    "files": file_summaries,
}

run_summary_file.parent.mkdir(exist_ok=True)
with run_summary_file.open("w", encoding="utf-8") as file:
    json.dump(run_summary, file, indent=2)

for file_name in processed_this_run:
    if file_name not in processed_files:
        processed_files.append(file_name)

manifest_file.parent.mkdir(exist_ok=True)
with manifest_file.open("w", encoding="utf-8") as file:
    json.dump(processed_files, file, indent=2)

logging.info("Pipeline finished")

2026-11-25 14:27:03,478 INFO: Starting pipeline
2026-11-25 14:27:03,478 INFO: Skipping already processed file: olist_customers_dataset.csv
2026-11-25 14:27:03,478 INFO: Skipping already processed file: olist_geolocation_dataset.csv
2026-11-25 14:27:03,490 INFO: Skipping already processed file: olist_order_items_dataset.csv
2026-11-25 14:27:03,492 INFO: Skipping already processed file: olist_orders_dataset.csv
2026-11-25 14:27:03,492 INFO: Skipping already processed file: olist_sellers_dataset.csv
2026-11-25 14:27:03,494 INFO: No new files to process
2026-11-25 14:27:03,497 INFO: Pipeline finished


## 8. Lint your .py modules with Ruff and fix warnings

In [9]:
!python -m ruff check ../day-12

All checks passed!
